# Lightspeed API Explorer

Interactive notebook for exploring the pyLightspeed library across all series.

| Series | Auth | Main endpoint used |
|---|---|---|
| **R-Series** | OAuth token file | `Items` |
| **C-Series** | Basic auth | `Products` |
| **X-Series** | Personal token | `Products` |

Credentials are loaded from the repo's `.env` file — fill that in before running.

## 1. Install & Import Required Libraries

In [1]:

# Add the src/ directory to sys.path so pylightspeed is importable.
# Dependencies are managed by uv — install any extras needed for this notebook.
import subprocess, sys
from pathlib import Path

REPO_ROOT = Path("C:/Data/Development/BottleManager/pyLightspeed")
SRC_DIR   = str(REPO_ROOT / "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Use uv to install into the active venv (uv venvs don't ship pip by default).
result = subprocess.run(
    ["uv", "add", "--python", sys.executable,
     "python-dotenv", "pandas",],
    capture_output=True,
    text=True,
)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"uv exited with code {result.returncode}")

print("Dependencies ready.")
print(f"pylightspeed src on path: {SRC_DIR}")


STDERR:
Resolved 53 packages in 1ms
Audited 16 packages in 1ms

Dependencies ready.
pylightspeed src on path: C:\Data\Development\BottleManager\pyLightspeed\src


In [2]:
import os, json, pprint
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from pylightspeed.api import (
    LightspeedRSeriesApi,
    LightspeedCSeriesApi,
    LightspeedXSeriesApi,
)

# Load .env from repo root (one level above this notebook)
load_dotenv(Path(".env"))

print("pylightspeed imported successfully.")

2026-03-10 12:39:50.773 | DEBUG    | pylightspeed.connection:<module>:45 - connection module loaded


pylightspeed imported successfully.


## 2. Configure Credentials

pyLightspeed reads credentials and tokens through a **TokenStore**. Run one or more options below — later options override earlier ones for any key that appears in both.

| Option | Best for |
|---|---|
| **A — `.env` / direct assignment** | Local dev; simplest setup |
| **B — Vault** | Production; secrets in HashiCorp Vault |
| **C — Composite** | Mix sources, or split read vs. write targets |

> **R-Series** — OAuth 2.0. Needs `client_id`, `client_secret`, and a token store that holds the rotating OAuth token.  
> **C-Series** — Basic Auth. Needs `api_key` + `api_secret`. No token lifecycle.  
> **X-Series** — Personal Access Token or OAuth.

In [3]:

# ── Option A: .env file / direct assignment ───────────────────────────────────
# Credentials are read from the .env already loaded in the imports cell above.
# Replace any os.getenv(...) call with a hard-coded string to override.

LSR_ACCOUNT_ID    = os.getenv("LSR_ACCOUNT_ID")
LSR_CLIENT_ID     = os.getenv("LSR_CLIENT_ID")
LSR_CLIENT_SECRET = os.getenv("LSR_CLIENT_SECRET")
LSR_TOKEN_FILE    = os.getenv("LSR_TOKEN_FILE")   # path where the OAuth token is stored on disk

LSC_API_KEY    = os.getenv("LSC_API_KEY")
LSC_API_SECRET = os.getenv("LSC_API_SECRET")
LSC_API_HOST   = os.getenv("LSC_API_HOST", "api.shoplightspeed.com")
LSC_API_PATH   = os.getenv("LSC_API_PATH", "/us/{}")

LSX_DOMAIN_PREFIX  = os.getenv("LSX_DOMAIN_PREFIX")
LSX_PERSONAL_TOKEN = os.getenv("LSX_PERSONAL_TOKEN")

print("Option A — environment / .env credentials:")
print(f"  R-Series account   : {LSR_ACCOUNT_ID   or '⚠ not set'}")
print(f"  R-Series token file: {LSR_TOKEN_FILE   or '⚠ not set'}")
print(f"  C-Series key       : {'set' if LSC_API_KEY else '⚠ not set'}")
print(f"  X-Series domain    : {LSX_DOMAIN_PREFIX or '⚠ not set'}")


Option A — environment / .env credentials:
  R-Series account   : ⚠ not set
  R-Series token file: ⚠ not set
  C-Series key       : ⚠ not set
  X-Series domain    : ⚠ not set


### Option B — VaultTokenStore

Reads credentials and the OAuth token from HashiCorp Vault KV v2.  
Requires `VAULT_URL` and `VAULT_TOKEN` in your `.env`.

- `credentials_path` — one or more Vault paths, merged left-to-right (later path overrides earlier)
- `token_path` — where the OAuth token is read from and written to (must be different from any credentials path)

In [4]:

# ── Option B: VaultTokenStore ─────────────────────────────────────────────────
from pylightspeed.connection import VaultTokenStore

VAULT_URL         = os.getenv("VAULT_URL", "http://localhost:8200")
VAULT_TOKEN       = os.getenv("VAULT_TOKEN")
VAULT_MOUNT_POINT = os.getenv("VAULT_MOUNT_POINT", "secret")

# Credentials: list merged left-to-right — shared keys first, store-specific last
VAULT_CREDENTIALS_PATHS = [
    os.getenv("VAULT_CREDENTIALS_SHARED_PATH", "lightspeed/shared"),
    os.getenv("VAULT_CREDENTIALS_STORE_PATH",  "lightspeed/stores/2/creds"),
]

# Token path must be separate from credentials paths (saves overwrite your credentials)
VAULT_TOKEN_PATH = os.getenv("VAULT_TOKEN_PATH", "lightspeed/stores/2/tokens")

try:
    vault_store = VaultTokenStore(
        token_path=VAULT_TOKEN_PATH,
        credentials_path=VAULT_CREDENTIALS_PATHS,
        vault_addr=VAULT_URL,
        vault_token=VAULT_TOKEN,
        mount_point=VAULT_MOUNT_POINT,
    )

    creds = vault_store.load_credentials()
    token = vault_store.load_token()

    print("Option B — Vault:")
    print(f"  URL             : {VAULT_URL}")
    print(f"  Credential keys : {sorted(creds.keys()) if creds else '(none loaded)'}")
    print(f"  Token present   : {'yes ✓' if token else 'no — complete OAuth setup below first'}")
    print()
    print("Usage:  lsr = LightspeedRSeriesApi(token_store=vault_store)")

except Exception as e:
    vault_store = None
    print(f"✗ Vault connection failed: {e}")
    print("  Check VAULT_URL and VAULT_TOKEN in your .env")


Option B — Vault:
  URL             : http://192.168.1.254:8200/
  Credential keys : ['LSC_API_HOST', 'LSC_API_KEY', 'LSC_API_PATH', 'LSC_API_SECRET', 'LSR_ACCOUNT_ID', 'LSR_API_HOST', 'LSR_API_PATH', 'LSR_CLIENT_ID', 'LSR_CLIENT_SECRET', 'LSR_REDIRECT_URI', 'STORE_NAME']
  Token present   : yes ✓

Usage:  lsr = LightspeedRSeriesApi(token_store=vault_store)


In [ ]:

# ── Option C: CompositeTokenStore — mix and match sources ─────────────────────
# Use when credentials and tokens live in different backends,
# or when you want token writes to go to multiple destinations.
#
# Uncomment and adapt whichever example fits your setup:

from pylightspeed.connection import CompositeTokenStore, FileTokenStore, EnvTokenStore

# -- Example 1: .env credentials + local token file
# composite_store = CompositeTokenStore(
#     credentials=EnvTokenStore(),
#     token_read=FileTokenStore(LSR_TOKEN_FILE),     # requires Option A above
#     token_write=FileTokenStore(LSR_TOKEN_FILE),
# )

# -- Example 2: Vault for everything, with a local file as a token backup
# composite_store = CompositeTokenStore(
#     credentials=vault_store,                       # requires Option B above
#     token_read=vault_store,
#     token_write=[vault_store, FileTokenStore("tokens.backup.json")],
# )

# -- Example 3: Merge .env (base) and Vault (store-specific override)
# composite_store = CompositeTokenStore(
#     credentials=[EnvTokenStore(), vault_store],    # vault values win on collision
#     token_read=vault_store,
#     token_write=vault_store,
# )

print("Uncomment one example above, then use:  lsr = LightspeedRSeriesApi(token_store=composite_store)")


## 3. First-time R-Series OAuth Setup

> **Skip this section if you already have a valid token.** If `vault_store` loaded a token above, or `LSR_TOKEN_FILE` exists on disk, you can go straight to section 4. YOU SHOULD NOT RUN THIS EVERY TIME! R-Series uses a refresh token (see their API docs) that only needs refreshed, not recreated. Read their docs closely.

R-Series uses OAuth 2.0. Do this once (or when a token has fully expired and can no longer auto-refresh):

1. Run the **Step 1** cell — opens Lightspeed's authorization page in your browser
2. Approve the connection
3. Copy the full redirect URL from the address bar
4. Paste it into `REDIRECT_URL` in the **Step 2** cell and run it

The token is saved automatically to `vault_store` (Option B) and/or `LSR_TOKEN_FILE` (Option A), whichever are configured.

In [ ]:

# ── OAuth Step 1 of 2: Generate the authorization URL ─────────────────────────
import webbrowser
from urllib.parse import parse_qs, urlparse
from pylightspeed.connection import RSeriesConnection

# Resolve credentials from vault_store (Option B) or .env constants (Option A)
_creds = {}
if globals().get("vault_store"):
    _creds = vault_store.load_credentials() or {}

_client_id     = _creds.get("LSR_CLIENT_ID")     or globals().get("LSR_CLIENT_ID")
_client_secret = _creds.get("LSR_CLIENT_SECRET") or globals().get("LSR_CLIENT_SECRET")
_redirect_uri  = (_creds.get("LSR_REDIRECT_URI") or os.getenv("LSR_REDIRECT_URI", "https://127.0.0.1/"))
_scope         = os.getenv("LSR_SCOPE", "employee:all")

if not all([_client_id, _client_secret]):
    raise ValueError(
        "LSR_CLIENT_ID and LSR_CLIENT_SECRET are required.\n"
        "Set them in .env (Option A) or in Vault (Option B) before running this cell."
    )

_auth_url, _oauth_state, _code_verifier = RSeriesConnection.get_authorization_url(
    _client_id, _scope, _redirect_uri
)

print("Opening Lightspeed authorization page…")
print(f"\n  {_auth_url}\n")
print("After approving, copy the full redirect URL from your browser's address bar")
print("and paste it into REDIRECT_URL in the next cell, then run that cell.")
webbrowser.open(_auth_url)


LSR_TOKEN_FILE not set — token will only be saved to Vault.
Opening Lightspeed authorization page…

  https://cloud.lightspeedapp.com/auth/oauth/authorize?response_type=code&client_id=577f37979925d3afa87ae7c870c34c126b856f8e6297acddb7a753a6fe005e74&scope=employee%3Aall&redirect_uri=https%3A%2F%2F127.0.0.1%3A5000&state=pIOvM-TF2naO6WOSgjdM-V_wCBLs9Xym&code_challenge=QgKK-1jGYnjwJmS5ToV0mje19k4f2SI24bCSCJiPVz4&code_challenge_method=S256

If the browser does not open, copy the URL above and paste it manually.

After approving, copy the full redirect URL from your browser's address bar
and paste it into REDIRECT_URL in the next cell, then run that cell.


True

In [ ]:
# ONLY RUN THIS CELL IF YOU HAVE ALREADY RUN THE PREVIOUS CELL TO GENERATE NEW CODES
# ── OAuth Step 2 of 2: Exchange code for token and save it ────────────────────
# Paste the full redirect URL from your browser's address bar below.

REDIRECT_URL = "paste-redirect-url-here"

if REDIRECT_URL == "paste-redirect-url-here":
    raise ValueError("Paste the redirect URL from your browser above, then re-run this cell.")

parsed = urlparse(REDIRECT_URL)
params = parse_qs(parsed.query)
code = params["code"][0]
returned_state = params.get("state", [None])[0]

if returned_state and returned_state != _oauth_state:
    raise ValueError(f"State mismatch — expected {_oauth_state!r}, got {returned_state!r}")

token_data = RSeriesConnection.exchange_code_for_token(
    _client_id, _client_secret, code, _redirect_uri,
    code_verifier=_code_verifier,
)

# ── Save token to whichever store(s) are configured ───────────────────────────
saved = False

# Option B: Vault
if globals().get("vault_store"):
    try:
        vault_store.save_token(token_data)
        print(f"✓ Token saved to Vault: {vault_store.token_path}")
        saved = True
    except Exception as e:
        print(f"  ⚠ Vault save failed: {e}")

# Option A: file
if globals().get("LSR_TOKEN_FILE"):
    from pylightspeed.connection import FileTokenStore
    FileTokenStore(LSR_TOKEN_FILE).save_token(token_data)
    print(f"✓ Token saved to file: {LSR_TOKEN_FILE}")
    saved = True

if not saved:
    print("⚠ Token not saved — no writable store configured.")
    print("  Set LSR_TOKEN_FILE in .env (Option A) or run the Vault cell (Option B).")

print()
print(f"  access_token present : {'access_token' in token_data}")
print(f"  refresh_token present: {'refresh_token' in token_data}")
print(f"  expires_in           : {token_data.get('expires_in', 'n/a')}s")
print("\nNow run the 'Create API Connections' cell.")


ValueError: Paste the redirect URL from your browser above, then re-run this cell.

## 4. Create API Connections

Once credentials and tokens are in place, create the API objects.

**Option i — TokenStore** (recommended): pass any store and it loads credentials + token automatically. One line per API.  
**Option ii — Direct assignment**: pass each credential explicitly — useful with Option A (`.env`).

Each object is set to `None` if its credentials cannot be resolved, so later cells can safely `if lsr:` guard before calling.

In [6]:

from pylightspeed.exception import MissingCredentialsError

lsr = lsc = lsx = None

# ── Option i: TokenStore ──────────────────────────────────────────────────────
# Pass a store and credentials + token are resolved automatically.
# Change vault_store to composite_store if you configured Option C above.

_ts = globals().get("vault_store")   # or: globals().get("composite_store")
if _ts:
    try:
        lsr = LightspeedRSeriesApi(token_store=_ts)
        print("✓ R-Series connected  (token store)")
    except MissingCredentialsError as e:
        print(f"✗ R-Series — {e}")

    try:
        lsc = LightspeedCSeriesApi(token_store=_ts)
        print("✓ C-Series connected  (token store)")
    except MissingCredentialsError as e:
        print(f"✗ C-Series — {e}")

    try:
        lsx = LightspeedXSeriesApi(token_store=_ts)
        print("✓ X-Series connected  (token store)")
    except MissingCredentialsError as e:
        print(f"✗ X-Series — {e}")

# ── Option ii: Direct assignment ──────────────────────────────────────────────
# Use this if you ran Option A (.env) only. Uncomment the series you need.

# if not lsr and all([globals().get(k) for k in ("LSR_ACCOUNT_ID","LSR_CLIENT_ID","LSR_CLIENT_SECRET","LSR_TOKEN_FILE")]):
#     lsr = LightspeedRSeriesApi(
#         account_id=LSR_ACCOUNT_ID,
#         client_id=LSR_CLIENT_ID,
#         client_secret=LSR_CLIENT_SECRET,
#         token_file=LSR_TOKEN_FILE,
#     )
#     print("✓ R-Series connected  (direct)")

# if not lsc and globals().get("LSC_API_KEY") and globals().get("LSC_API_SECRET"):
#     lsc = LightspeedCSeriesApi(api_key=LSC_API_KEY, api_secret=LSC_API_SECRET,
#                                host=LSC_API_HOST, api_path=LSC_API_PATH)
#     print("✓ C-Series connected  (direct)")

# if not lsx and globals().get("LSX_DOMAIN_PREFIX") and globals().get("LSX_PERSONAL_TOKEN"):
#     lsx = LightspeedXSeriesApi(domain_prefix=LSX_DOMAIN_PREFIX, personal_token=LSX_PERSONAL_TOKEN)
#     print("✓ X-Series connected  (direct)")

if not any([lsr, lsc, lsx]):
    print("No APIs connected.")
    print("  • Token store path: run Option B (Vault) or Option C (Composite) above")
    print("  • Direct path: uncomment Option ii blocks and run Option A (.env) first")


✓ R-Series connected  (token store)
✓ C-Series connected  (token store)
✗ X-Series — LightspeedXSeriesApi is missing required credentials: domain_prefix + personal_token (personal token auth), OR client_id + token_store (OAuth). Provide them as constructor arguments or via a TokenStore that implements load_credentials().


## 3. Get a List of Items / Products

`page()` returns the first page of results (up to 100 for R-Series, 50 for C/X-Series by default).  
Pass `limit=N` to control page size; use `listall()` to fetch every page automatically.

### R-Series — Items

In [7]:
if lsr:
    # Fetch first page (default limit = 100 for R-Series)
    rsr_items = lsr.Items.page(limit=10)
    print(f"Returned {len(rsr_items)} items  |  total in store: {lsr.connection.count}")
    print()

    # Build a tidy summary table from the scalar fields
    rows = [item.as_dict() for item in rsr_items]
    df_rsr = pd.DataFrame(rows)

    # Show a curated subset of columns if they exist
    show_cols = [c for c in ["itemID", "description", "sku", "createTime", "timeStamp"]
                 if c in df_rsr.columns]
    display(df_rsr[show_cols].head(10))
else:
    print("R-Series not configured — skipping.")

Returned 10 items  |  total in store: 9879



,itemID,description,createTime,timeStamp
0,4,1000 Stories-Gold Rush-Red Wine-2018-750ml,2020-09-29T20:00:06+00:00,2024-08-02T22:40:50+00:00
1,6,1000 Stories Zinfandel,2020-09-29T20:00:06+00:00,2024-06-24T14:04:16+00:00
2,12,1757 Vermouth Extra Dry | 1l,2020-09-29T20:59:32+00:00,2024-09-29T19:03:12+00:00
3,14,Barton 1792 Small Batch Bourbon 1.75l,2020-09-29T20:59:32+00:00,2025-02-28T22:46:25+00:00
4,15,Barton 1792 Small Batch Bourbon 750ml,2020-09-29T14:59:32+00:00,2026-03-10T12:53:28+00:00
5,16,1792 Bottle In Bond | bottles,2020-09-29T20:59:32+00:00,2024-09-29T18:55:23+00:00
6,17,1792 Bottled And Bond,2020-09-29T20:59:32+00:00,2026-03-10T11:26:19+00:00
7,18,1792 Full Proof Single Barrel | 750ml,2020-09-29T20:59:32+00:00,2026-03-10T11:26:05+00:00
8,21,1792 Single Barrel 750ml,2020-09-29T14:59:32+00:00,2026-03-10T12:52:39+00:00
9,22,1792 Sweet Wheat,2020-09-29T14:59:32+00:00,2026-03-10T11:25:54+00:00


In [ ]:
item1=lsr.Items.get(1)
item1

In [ ]:
item1.json

### C-Series — Products

In [8]:
if lsc:
    # C-Series default page size is 50; max is 250
    lsc_products = lsc.Products.page(limit=10)
    print(f"Returned {len(lsc_products)} products")
    print()

    rows = [prod.as_dict() for prod in lsc_products]
    df_lsc = pd.DataFrame(rows)

    show_cols = [c for c in ["id", "title", "sku", "price", "isVisible", "createdAt", "updatedAt"]
                 if c in df_lsc.columns]
    display(df_lsc[show_cols].head(10))
else:
    print("C-Series not configured — skipping.")

Returned 10 products



,id,title,isVisible,createdAt,updatedAt
0,69679394,Another Hendricks Gin 750ml,False,2026-03-07T17:29:45+00:00,2026-03-07T17:47:04+00:00
1,69594917,Shortbarrel After The Swarm Rye 750ml,False,2026-02-27T22:53:03+00:00,2026-03-07T16:26:15+00:00
2,69589969,Foxes Bow Irish Whiskey 750ml,False,2026-02-27T17:51:58+00:00,2026-02-27T19:38:30+00:00
3,69587336,EG Windsor Earl Grey & Sage Flavored Vodka 750ml,False,2026-02-27T14:45:56+00:00,2026-02-27T14:49:16+00:00
4,69515349,Sullivans Cove Small Batch French Oa,False,2026-02-19T02:45:19+00:00,2026-02-19T02:45:23+00:00
5,69451591,Zaya Gran Reserva Rum 750ml,False,2026-02-11T22:25:48+00:00,2026-02-11T22:26:04+00:00
6,69451490,Templeton 4 Year Rye 750ml,False,2026-02-11T22:18:35+00:00,2026-02-11T22:18:36+00:00
7,69351071,Dry Town Gin 750ml,False,2026-01-31T23:10:43+00:00,2026-01-31T23:10:49+00:00
8,69055505,GlenDronach Ode To The Embers 750ml,False,2025-12-20T20:33:49+00:00,2026-03-01T04:53:22+00:00
9,69055495,GlenDronach Ode To The Dark 750ml,False,2025-12-20T20:30:36+00:00,2026-02-28T16:29:26+00:00


### X-Series — Products

In [ ]:
if lsx:
    lsx_products = lsx.Products.page()
    print(f"Returned {len(lsx_products)} products")
    print()

    rows = [prod.as_dict() for prod in lsx_products]
    df_lsx = pd.DataFrame(rows)

    show_cols = [c for c in ["id", "name", "sku", "base_price", "retail_price",
                              "created_at", "updated_at"]
                 if c in df_lsx.columns]
    display(df_lsx[show_cols].head(10))
else:
    print("X-Series not configured — skipping.")

## 4. Get a Single Item

`get(id)` fetches one record by its primary key and returns a full object with dot-access to every field.

**Before running the update cells below**, set `RSR_ITEM_ID`, `LSC_PRODUCT_ID`, and/or `LSX_PRODUCT_ID` to a real ID from your store.  
The cells here will auto-populate them from the list results above.

### R-Series — single Item

In [10]:
rsr_item = None

if lsr:
    # Use the first item from the list above, or override with a specific ID:
    #   RSR_ITEM_ID = "1234"
    RSR_ITEM_ID = rsr_items[0]["itemID"] if rsr_items else None

    if RSR_ITEM_ID:
        rsr_item = lsr.Items.get(RSR_ITEM_ID)

        print(f"Item ID : {rsr_item.itemID}")
        print(f"Name    : {rsr_item.description}")
        # Note: use dict.get() — ApiResource.get() is the HTTP fetch classmethod,
        # which shadows dict.get on instances.
        print(f"SKU     : {dict.get(rsr_item, 'customSku', '—')}")
        print()

        # Prices are nested — nested_json_to_attr() populates the convenience fields
        rsr_item.nested_json_to_attr()
        print(f"Default price : {rsr_item.price_default}")
        print(f"MSRP          : {rsr_item.price_msrp}")
        print(f"Online price  : {rsr_item.price_online}")
        print()

        # Show all top-level keys returned by the API
        print("All fields returned:")
        pprint.pprint({k: v for k, v in rsr_item.items() if not k.startswith("_")})
    else:
        print("No items found to inspect.")
else:
    print("R-Series not configured — skipping.")


Item ID : 4
Name    : 1000 Stories-Gold Rush-Red Wine-2018-750ml
SKU     : 082896001620

Default price : 10.99
MSRP          : 0
Online price  : 0

All fields returned:
{'Prices': {'ItemPrice': [{'amount': '10.99',
                           'useType': 'Default',
                           'useTypeID': '1'},
                          {'amount': '0', 'useType': 'MSRP', 'useTypeID': '2'},
                          {'amount': '0',
                           'useType': 'Online',
                           'useTypeID': '3'},
                          {'amount': '10.99',
                           'useType': 'Promotion',
                           'useTypeID': '4'}]},
 'archived': 'false',
 'avgCost': '10.95',
 'categoryID': '102',
 'createTime': '2020-09-29T20:00:06+00:00',
 'customSku': '082896001620',
 'defaultCost': '10.95',
 'defaultVendorID': '2',
 'departmentID': '0',
 'description': '1000 Stories-Gold Rush-Red Wine-2018-750ml',
 'discountable': 'true',
 'ean': '',
 'itemID': '4',
 'i

### C-Series — single Product

In [11]:
lsc_product = None

if lsc:
    # Use the first product from the list above, or override:
    #   LSC_PRODUCT_ID = 58526124
    LSC_PRODUCT_ID = lsc_products[0]["id"] if lsc_products else None

    if LSC_PRODUCT_ID:
        lsc_product = lsc.Products.get(LSC_PRODUCT_ID)

        print(f"Product ID  : {lsc_product.id}")
        print(f"Title       : {lsc_product.title}")
        print(f"SKU         : {dict.get(lsc_product, 'sku', '—')}")
        print(f"Price       : {dict.get(lsc_product, 'price', '—')}")
        print(f"Visible     : {dict.get(lsc_product, 'isVisible', '—')}")
        print()

        print("All fields returned:")
        pprint.pprint({k: v for k, v in lsc_product.items() if not k.startswith("_")})
    else:
        print("No products found to inspect.")
else:
    print("C-Series not configured — skipping.")


Product ID  : 69679394
Title       : Another Hendricks Gin 750ml
SKU         : —
Price       : —
Visible     : False

All fields returned:
{'attributes': {'resource': {'id': False,
                             'link': 'https://api.shoplightspeed.com/us/products/69679394/attributes.json',
                             'url': 'products/69679394/attributes'}},
 'brand': {'resource': {'id': 1970576,
                        'link': 'https://api.shoplightspeed.com/us/brands/1970576.json',
                        'url': 'brands/1970576'}},
 'categories': {'resource': {'id': False,
                             'link': 'https://api.shoplightspeed.com/us/categories/products.json?product=69679394',
                             'url': 'categories/products?product=69679394'}},
 'content': '',
 'createdAt': '2026-03-07T17:29:45+00:00',
 'data01': '',
 'data02': '',
 'data03': '',
 'deliverydate': False,
 'description': '',
 'fulltitle': 'Another Hendricks Gin 750ml',
 'hasMatrix': False,
 'id': 69679

### X-Series — single Product

In [ ]:
lsx_product = None

if lsx:
    # Use the first product from the list above, or override:
    #   LSX_PRODUCT_ID = "abc-123"
    LSX_PRODUCT_ID = lsx_products[0]["id"] if lsx_products else None

    if LSX_PRODUCT_ID:
        lsx_product = lsx.Products.get(LSX_PRODUCT_ID)

        print(f"Product ID   : {lsx_product.id}")
        print(f"Name         : {dict.get(lsx_product, 'name', '—')}")
        print(f"SKU          : {dict.get(lsx_product, 'sku', '—')}")
        print(f"Base price   : {dict.get(lsx_product, 'base_price', '—')}")
        print(f"Retail price : {dict.get(lsx_product, 'retail_price', '—')}")
        print()

        print("All fields returned:")
        pprint.pprint({k: v for k, v in lsx_product.items() if not k.startswith("_")})
    else:
        print("No products found to inspect.")
else:
    print("X-Series not configured — skipping.")


## 5. Update Item Name

`item.update(**fields)` sends a PUT request with the changed fields and returns the updated object.

> ⚠️ **These cells write to your live store.**  
> Check the item ID and new value before running.

- R-Series name field: `description`  
- C-Series name field: `title`  
- X-Series name field: `name`

### R-Series — update Item name

In [ ]:
if lsr and rsr_item:
    NEW_NAME = rsr_item.description + " (updated)"  # ← change this to whatever you want

    print(f"Before: {rsr_item.description!r}")

    updated = rsr_item.update(description=NEW_NAME)

    print(f"After : {updated.description!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("R-Series not configured or no item loaded — skipping.")

### C-Series — update Product title

In [ ]:
if lsc and lsc_product:
    NEW_TITLE = lsc_product.title + " (updated)"  # ← change this

    print(f"Before: {lsc_product.title!r}")

    updated = lsc_product.update(title=NEW_TITLE)

    print(f"After : {updated.title!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("C-Series not configured or no product loaded — skipping.")

### X-Series — update Product name

In [ ]:
if lsx and lsx_product:
    NEW_NAME_X = dict.get(lsx_product, "name", "") + " (updated)"  # ← change this

    print(f"Before: {dict.get(lsx_product, 'name')!r}")

    updated = lsx_product.update(name=NEW_NAME_X)

    print(f"After : {dict.get(updated, 'name')!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("X-Series not configured or no product loaded — skipping.")


### R-Series — update Item price

In [ ]:
if lsr and rsr_item:
    # Re-fetch to get the current price structure before editing
    rsr_item = lsr.Items.get(rsr_item.itemID)
    rsr_item.nested_json_to_attr()

    NEW_DEFAULT_PRICE = "19.99"   # ← set your new price
    NEW_MSRP          = "24.99"   # ← set your new MSRP

    print(f"Before — Default: {rsr_item.price_default}  MSRP: {rsr_item.price_msrp}")

    # R-Series requires you to send back ALL price tiers, not just the one you want to change.
    # Pull the existing tiers and patch the ones you want.
    item_prices = rsr_item["Prices"]["ItemPrice"]   # list of price dicts
    item_prices[0]["amount"] = NEW_DEFAULT_PRICE    # useTypeID 1: Default
    item_prices[1]["amount"] = NEW_MSRP             # useTypeID 2: MSRP

    updated = rsr_item.update(Prices={"ItemPrice": item_prices})
    updated.nested_json_to_attr()

    print(f"After  — Default: {updated.price_default}  MSRP: {updated.price_msrp}")
    print()
    print("Full updated Prices block:")
    pprint.pprint(updated["Prices"])
else:
    print("R-Series not configured or no item loaded — skipping.")

### C-Series — update Product price

In [ ]:
if lsc and lsc_product:
    NEW_PRICE = 19.99   # ← set your new price (float, not string)

    print(f"Before: {dict.get(lsc_product, 'price', '—')!r}")

    updated = lsc_product.update(price=NEW_PRICE)

    print(f"After : {dict.get(updated, 'price', '—')!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("C-Series not configured or no product loaded — skipping.")


### X-Series — update Product price

In [ ]:
if lsx and lsx_product:
    NEW_BASE_PRICE   = 19.99   # ← set your new base price
    NEW_RETAIL_PRICE = 24.99   # ← set your new retail price

    print(f"Before — base: {dict.get(lsx_product, 'base_price')!r}  "
          f"retail: {dict.get(lsx_product, 'retail_price')!r}")

    updated = lsx_product.update(
        base_price=NEW_BASE_PRICE,
        retail_price=NEW_RETAIL_PRICE,
    )

    print(f"After  — base: {dict.get(updated, 'base_price')!r}  "
          f"retail: {dict.get(updated, 'retail_price')!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("X-Series not configured or no product loaded — skipping.")


---

## Tips & further exploration

### Listing all resources (pagination handled automatically)

```python
all_items = lsr.Items.listall()           # R-Series — blocks until all pages fetched
all_products = lsc.Products.listall()     # C-Series
```

### Streaming large catalogues with a generator

```python
for item in lsr.Items.iter(limit=100):    # R-Series — yields one item at a time
    print(item.itemID, item.description)

for prod in lsc.Products.iterall():       # C-Series
    print(prod.id, prod.title)
```

### Filtering results

```python
# R-Series — filter by timeStamp (incremental sync)
recent = lsr.Items.page(timeStamp=">2026-01-01T00:00:00+00:00", limit=100)

# C-Series — filter by updated date
recent = lsc.Products.page(updated_at_min="2026-01-01 00:00:00", limit=50)
```

### Counting records (C-Series only)

```python
total = lsc.Products.count()
print(f"Total products: {total}")
```

### Accessing the raw JSON

Every object has a `json` key with the raw dict that came from the API:

```python
print(rsr_item["json"])
print(lsc_product["json"])
```

### Rate limits

The connection object tracks rate-limit state automatically.  
You can inspect the last response for debugging:

```python
print(lsr.connection._last_response.headers)
```